In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
master = pd.read_csv(r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Data\master_tourism_dataset.csv")

In [2]:
# Create user-item matrix
user_item_matrix = master.pivot_table(
    index='UserId',
    columns='AttractionId',
    values='Rating'
)

user_item_matrix.head()


AttractionId,369,481,640,650,673,737,748,749,824,841,...,1133,1137,1166,1171,1220,1225,1238,1278,1280,1297
UserId,,,,,,,,,,,,,,,,,,,,,
14,NaN,NaN,4.00,NaN,NaN,NaN,5.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,NaN,5.0,4.25,NaN,NaN,NaN,5.0,NaN,5.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
user_item_matrix_filled = user_item_matrix.fillna(0)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_item_matrix_filled)

import pandas as pd

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

user_similarity_df.head()


UserId,14,16,20,23,25,26,27,28,29,32,...,88168,88170,88175,88177,88178,88179,88185,88187,88189,88190
UserId,,,,,,,,,,,,,,,,,,,,,
14,1.000000,0.603672,0.000000,0.780869,0.0,0.780869,0.624695,0.0,0.624695,0.624695,...,0.624695,0.404029,0.624695,0.624695,0.624695,0.780869,0.327429,0.000000,0.499756,0.780869
16,0.603672,1.000000,0.460165,0.460165,0.0,0.460165,0.391141,0.0,0.391141,0.391141,...,0.391141,0.252975,0.391141,0.391141,0.391141,0.460165,0.578861,0.460165,0.312913,0.460165
20,0.000000,0.460165,1.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.314485,0.000000,0.000000,0.000000
23,0.780869,0.460165,0.000000,1.000000,0.0,1.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.419314,0.000000,0.000000,1.000000
25,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [15]:
def recommend_attractions(user_id, top_n=5):
    
    # Similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    
    # Exclude self
    similar_users = similar_users.iloc[1:11]
    
    # Get attractions rated by similar users
    similar_user_ids = similar_users.index
    
    similar_users_ratings = user_item_matrix.loc[similar_user_ids]
    
    # Average ratings from similar users
    recommended_scores = similar_users_ratings.mean().sort_values(ascending=False)
    
    # Remove attractions already rated by user
    user_rated = user_item_matrix.loc[user_id].dropna().index
    
    recommendations = recommended_scores.drop(user_rated)
    
    return recommendations.head(top_n)


In [16]:
sample_user = master['UserId'].iloc[0]

recommend_attractions(sample_user)


AttractionId
369   NaN
481   NaN
650   NaN
673   NaN
737   NaN
dtype: float64

In [17]:
def recommend_attraction_names(user_id, top_n=5):
    
    rec_ids = recommend_attractions(user_id, top_n).index
    
    return master[master['AttractionId'].isin(rec_ids)][
        ['AttractionId','Attraction']
    ].drop_duplicates()


In [18]:
recommend_attraction_names(sample_user)


,AttractionId,Attraction
19626,673,Seminyak Beach
22538,481,Nusa Dua Beach
24642,650,Sanur Beach
39045,737,Tanah Lot Temple
42397,369,Kuta Beach - Bali


In [19]:
def recommend_attraction_names(user_id, top_n=5):
    
    rec_scores = recommend_attractions(user_id, top_n)
    
    rec_df = pd.DataFrame({
        'AttractionId': rec_scores.index,
        'Score': rec_scores.values
    })
    
    rec_df = rec_df.merge(
        master[['AttractionId', 'Attraction']],
        on='AttractionId',
        how='left'
    ).drop_duplicates()
    
    return rec_df[['Attraction', 'Score']]


In [20]:
recommend_attraction_names(sample_user)


,Attraction,Score
0,Kuta Beach - Bali,NaN
2765,Nusa Dua Beach,NaN
4869,Sanur Beach,NaN
7912,Seminyak Beach,NaN
10824,Tanah Lot Temple,NaN


In [21]:
master[master['UserId'] == sample_user]['AttractionType'].value_counts()


AttractionType
Nature & Wildlife Areas    1
Name: count, dtype: int64

In [22]:
attraction_lookup = master[['AttractionId', 'Attraction']].drop_duplicates()


In [23]:
def recommend_attraction_names(user_id, top_n=5):
    
    rec_scores = recommend_attractions(user_id, top_n)
    
    rec_df = pd.DataFrame({
        'AttractionId': rec_scores.index,
        'Score': rec_scores.values
    })
    
    rec_df = rec_df.merge(
        attraction_lookup,
        on='AttractionId',
        how='left'
    )
    
    return rec_df[['Attraction', 'Score']]


In [24]:
recommend_attraction_names(sample_user)


,Attraction,Score
0,Kuta Beach - Bali,NaN
1,Nusa Dua Beach,NaN
2,Sanur Beach,NaN
3,Seminyak Beach,NaN
4,Tanah Lot Temple,NaN


In [25]:
def recommend_attractions(user_id, top_n=5):
    
    # Use filled matrix
    matrix = user_item_matrix.fillna(0)
    
    # Compute similarity fresh (safe version)
    from sklearn.metrics.pairwise import cosine_similarity
    similarity = cosine_similarity(matrix)
    
    similarity_df = pd.DataFrame(
        similarity,
        index=matrix.index,
        columns=matrix.index
    )
    
    # Similar users
    similar_users = similarity_df[user_id].sort_values(ascending=False)
    similar_users = similar_users.iloc[1:11]  # top 10 similar
    
    similar_user_ids = similar_users.index
    
    # Use filled matrix here
    similar_users_ratings = matrix.loc[similar_user_ids]
    
    # Average ratings
    recommended_scores = similar_users_ratings.mean()
    
    # Remove already rated
    user_rated = matrix.loc[user_id]
    recommended_scores = recommended_scores[user_rated == 0]
    
    return recommended_scores.sort_values(ascending=False).head(top_n)


In [26]:
attraction_lookup = master[['AttractionId', 'Attraction']].drop_duplicates()


In [27]:
def recommend_attraction_names(user_id, top_n=5):
    
    rec_scores = recommend_attractions(user_id, top_n)
    
    rec_df = pd.DataFrame({
        'AttractionId': rec_scores.index,
        'Score': rec_scores.values
    })
    
    rec_df = rec_df.merge(
        attraction_lookup,
        on='AttractionId',
        how='left'
    )
    
    return rec_df[['Attraction', 'Score']]


In [30]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_attractions(user_id, top_n=5):
    
    # Fill missing values
    matrix = user_item_matrix.fillna(0)
    
    # Get target user vector
    user_vector = matrix.loc[user_id].values.reshape(1, -1)
    
    # Compute similarity only between this user and all others
    similarity = cosine_similarity(user_vector, matrix)[0]
    
    similarity_series = pd.Series(similarity, index=matrix.index)
    
    # Remove self similarity
    similarity_series = similarity_series.drop(user_id)
    
    # Get top similar users
    top_users = similarity_series.sort_values(ascending=False).head(10).index
    
    # Get their ratings
    similar_users_ratings = matrix.loc[top_users]
    
    # Compute mean score
    recommended_scores = similar_users_ratings.mean()
    
    # Remove attractions already rated
    user_rated = matrix.loc[user_id]
    recommended_scores = recommended_scores[user_rated == 0]
    
    return recommended_scores.sort_values(ascending=False).head(top_n)


In [31]:
def recommend_attraction_names(user_id, top_n=5):
    
    rec_scores = recommend_attractions(user_id, top_n)
    
    rec_df = pd.DataFrame({
        'AttractionId': rec_scores.index,
        'Score': rec_scores.values
    })
    
    rec_df = rec_df.merge(
        attraction_lookup,
        on='AttractionId',
        how='left'
    )
    
    return rec_df[['Attraction', 'Score']]


In [32]:
recommend_attraction_names(sample_user)


,Attraction,Score
0,Kuta Beach - Bali,0.0
1,Nusa Dua Beach,0.0
2,Sanur Beach,0.0
3,Seminyak Beach,0.0
4,Tanah Lot Temple,0.0


In [33]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_attractions(user_id, top_n=5):
    
    matrix = user_item_matrix.fillna(0)
    
    # Target user vector
    user_vector = matrix.loc[user_id].values.reshape(1, -1)
    
    # Compute similarity to all users
    similarity = cosine_similarity(user_vector, matrix)[0]
    
    similarity_series = pd.Series(similarity, index=matrix.index)
    
    # Remove self
    similarity_series = similarity_series.drop(user_id)
    
    # Get top similar users
    top_users = similarity_series.sort_values(ascending=False).head(10)
    
    # Get their ratings
    similar_users_ratings = matrix.loc[top_users.index]
    
    # Weighted average
    weighted_scores = np.dot(top_users.values, similar_users_ratings)
    similarity_sum = np.sum(top_users.values)
    
    if similarity_sum == 0:
        return pd.Series(dtype=float)
    
    recommended_scores = weighted_scores / similarity_sum
    
    recommended_scores = pd.Series(
        recommended_scores,
        index=matrix.columns
    )
    
    # Remove already rated
    user_rated = matrix.loc[user_id]
    recommended_scores = recommended_scores[user_rated == 0]
    
    return recommended_scores.sort_values(ascending=False).head(top_n)


In [34]:
recommend_attraction_names(sample_user)


,Attraction,Score
0,Kuta Beach - Bali,0.0
1,Nusa Dua Beach,0.0
2,Sanur Beach,0.0
3,Seminyak Beach,0.0
4,Tanah Lot Temple,0.0


In [35]:
print("Average ratings per user:",
      master.groupby('UserId')['AttractionId'].count().mean())

print("Average ratings per attraction:",
      master.groupby('AttractionId')['UserId'].count().mean())


Average ratings per user: 1.5785360615641593
Average ratings per attraction: 1764.0666666666666


In [36]:
def get_user_preferred_types(user_id):
    
    user_data = master[master['UserId'] == user_id]
    
    liked = user_data[user_data['Rating'] >= 4]
    
    return liked['AttractionType'].value_counts().index.tolist()


In [37]:
def content_based_recommendation(user_id, top_n=5):
    
    preferred_types = get_user_preferred_types(user_id)
    
    if not preferred_types:
        return pd.DataFrame()
    
    # Attractions user already visited
    visited = master[master['UserId'] == user_id]['AttractionId']
    
    # Candidate attractions
    candidates = master[
        (master['AttractionType'].isin(preferred_types)) &
        (~master['AttractionId'].isin(visited))
    ]
    
    # Rank by popularity + average rating
    ranked = candidates.groupby(
        ['AttractionId', 'Attraction']
    ).agg({
        'Rating': 'mean',
        'AttractionPopularity': 'mean'
    }).reset_index()
    
    ranked['Score'] = ranked['Rating'] * 0.7 + ranked['AttractionPopularity'] * 0.3
    
    return ranked.sort_values(by='Score', ascending=False).head(top_n)[
        ['Attraction', 'Score']
    ]


In [38]:
content_based_recommendation(sample_user)


,Attraction,Score
0,Sempu Island,18.845283


In [39]:
from sklearn.metrics.pairwise import cosine_similarity

# Use filled matrix
matrix = user_item_matrix.fillna(0)

# Transpose for item similarity
item_similarity = cosine_similarity(matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=matrix.columns,
    columns=matrix.columns
)


In [40]:
def item_based_recommendation(user_id, top_n=5):
    
    matrix = user_item_matrix.fillna(0)
    
    user_ratings = matrix.loc[user_id]
    
    # Attractions rated by user
    rated_items = user_ratings[user_ratings > 0]
    
    if rated_items.empty:
        return pd.DataFrame()
    
    scores = {}
    
    for item_id, rating in rated_items.items():
        
        similar_items = item_similarity_df[item_id]
        
        for sim_item, similarity in similar_items.items():
            
            if user_ratings[sim_item] == 0:
                
                if sim_item not in scores:
                    scores[sim_item] = 0
                
                scores[sim_item] += similarity * rating
    
    rec_df = pd.DataFrame(
        list(scores.items()),
        columns=['AttractionId', 'Score']
    )
    
    rec_df = rec_df.merge(
        attraction_lookup,
        on='AttractionId',
        how='left'
    )
    
    return rec_df.sort_values(by='Score', ascending=False).head(top_n)[
        ['Attraction', 'Score']
    ]


In [41]:
sample_user = master['UserId'].iloc[0]

In [42]:
item_based_recommendation(sample_user)


,Attraction,Score
5,Tegalalang Rice Terrace,1.370016
6,Tegenungan Waterfall,0.873101
4,Tanah Lot Temple,0.582516
8,Waterbom Bali,0.552904
7,Uluwatu Temple,0.520464


In [43]:
attraction_features = master.groupby(
    ['AttractionId', 'Attraction', 'AttractionType']
).agg({
    'AttractionPopularity': 'mean',
    'Rating': 'mean'
}).reset_index()

attraction_features.rename(columns={
    'Rating': 'AvgRating'
}, inplace=True)



In [44]:
from sklearn.preprocessing import LabelEncoder

le_type = LabelEncoder()
attraction_features['TypeEncoded'] = le_type.fit_transform(
    attraction_features['AttractionType']
)


In [45]:
content_matrix = attraction_features[
    ['TypeEncoded', 'AttractionPopularity', 'AvgRating']
]


In [46]:
from sklearn.metrics.pairwise import cosine_similarity

content_similarity = cosine_similarity(content_matrix)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=attraction_features['AttractionId'],
    columns=attraction_features['AttractionId']
)


In [47]:
def content_based_recommendation(user_id, top_n=5):
    
    matrix = user_item_matrix.fillna(0)
    
    user_ratings = matrix.loc[user_id]
    
    rated_items = user_ratings[user_ratings > 0].index
    
    scores = {}
    
    for item in rated_items:
        
        similar_items = content_similarity_df[item]
        
        for sim_item, similarity in similar_items.items():
            
            if user_ratings[sim_item] == 0:
                
                if sim_item not in scores:
                    scores[sim_item] = 0
                
                scores[sim_item] += similarity
    
    rec_df = pd.DataFrame(
        list(scores.items()),
        columns=['AttractionId', 'Score']
    )
    
    rec_df = rec_df.merge(
        attraction_lookup,
        on='AttractionId',
        how='left'
    )
    
    return rec_df.sort_values(by='Score', ascending=False).head(top_n)[
        ['Attraction', 'Score']
    ]


In [48]:
content_based_recommendation(sample_user)


,Attraction,Score
0,Kuta Beach - Bali,1.000000
2,Sanur Beach,1.000000
3,Seminyak Beach,1.000000
5,Tegalalang Rice Terrace,0.999999
1,Nusa Dua Beach,0.999998


In [49]:
attraction_features = master.groupby(
    ['AttractionId', 'Attraction', 'AttractionType']
).agg({
    'AttractionPopularity': 'mean',
    'Rating': 'mean'
}).reset_index()

attraction_features.rename(columns={
    'Rating': 'AvgRating'
}, inplace=True)

# One-hot encoding
type_dummies = pd.get_dummies(
    attraction_features['AttractionType'],
    prefix='Type'
)

attraction_features = pd.concat(
    [attraction_features, type_dummies],
    axis=1
)


In [50]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

attraction_features[['AttractionPopularity', 'AvgRating']] = scaler.fit_transform(
    attraction_features[['AttractionPopularity', 'AvgRating']]
)


In [51]:
feature_columns = list(type_dummies.columns) + [
    'AttractionPopularity',
    'AvgRating'
]

content_matrix = attraction_features[feature_columns]


In [52]:
from sklearn.metrics.pairwise import cosine_similarity

content_similarity = cosine_similarity(content_matrix)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=attraction_features['AttractionId'],
    columns=attraction_features['AttractionId']
)


In [53]:
content_based_recommendation(sample_user)


,Attraction,Score
18,Sempu Island,0.768884
8,Waterbom Bali,0.489388
5,Tegalalang Rice Terrace,0.426172
7,Uluwatu Temple,0.354699
4,Tanah Lot Temple,0.350561


In [54]:
def hybrid_recommendation(user_id, top_n=5, alpha=0.6):
    """
    alpha = weight for collaborative filtering
    (1-alpha) = weight for content-based
    """
    
    # Get CF scores
    cf_scores = item_based_recommendation(user_id, top_n=50)
    cf_scores = cf_scores.merge(
        attraction_lookup,
        on='Attraction',
        how='left'
    )
    
    # Get Content scores
    cb_scores = content_based_recommendation(user_id, top_n=50)
    cb_scores = cb_scores.merge(
        attraction_lookup,
        on='Attraction',
        how='left'
    )
    
    # Merge both
    hybrid_df = pd.merge(
        cf_scores[['Attraction', 'Score']],
        cb_scores[['Attraction', 'Score']],
        on='Attraction',
        how='outer',
        suffixes=('_cf', '_cb')
    ).fillna(0)
    
    # Weighted score
    hybrid_df['FinalScore'] = (
        alpha * hybrid_df['Score_cf'] +
        (1 - alpha) * hybrid_df['Score_cb']
    )
    
    return hybrid_df.sort_values(
        by='FinalScore',
        ascending=False
    ).head(top_n)[['Attraction', 'FinalScore']]


In [55]:
hybrid_recommendation(sample_user)


,Attraction,FinalScore
22,Tegalalang Rice Terrace,0.992479
23,Tegenungan Waterfall,0.621627
27,Waterbom Bali,0.527498
21,Tanah Lot Temple,0.489734
25,Uluwatu Temple,0.454158


In [57]:
import joblib

In [58]:
joblib.dump(user_item_matrix, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Recomendations\user_item_matrix.pkl")
joblib.dump(item_similarity_df, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Recomendations\item_similarity.pkl")
joblib.dump(content_similarity_df, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Recomendations\content_similarity.pkl")
joblib.dump(attraction_lookup, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Recomendations\attraction_lookup.pkl")


['C:\\Users\\kisho\\Downloads\\Tourism_Experience_Analytics_Project\\Recomendations\\attraction_lookup.pkl']